# Phase 2 — 앱스토어 리뷰 수집

**수집 대상**
| 앱 | 스토어 | 국가 |
|---|---|---|
| LOCA EV | Google Play / App Store | 라오스 (LAO) |
| Green SM | Google Play / App Store | 라오스 (LAO) |
| V-Green | Google Play / App Store | 베트남 (VNM) |

**파이프라인**
```
google-play-scraper / app-store-scraper
         ↓
    DataFrame 정제
         ↓
  Supabase app_reviews 테이블
```

⚠️ **주의**: 수집 전 앱별 리뷰 수를 먼저 확인하세요. LOCA EV는 현지 앱으로 리뷰가 적을 수 있습니다.

In [ ]:
# 필요 라이브러리 설치
# !pip install google-play-scraper app-store-scraper psycopg2-binary pandas python-dotenv

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
from datetime import datetime
from google_play_scraper import reviews, Sort
from app_store_scraper import AppStore
from src.db import insert_df

## 1. 수집 대상 앱 정의

In [ ]:
# 수집 대상 앱 목록
# ⚠️ App ID는 실제 스토어에서 확인 필요
APPS = [
    {
        "app_name": "LOCA EV",
        "country": "LAO",
        "google_play_id": "com.loca.ev",          # 실제 ID로 교체 필요
        "app_store_id": None,                       # 없을 수 있음
        "app_store_country": "la",
    },
    {
        "app_name": "Green SM",
        "country": "LAO",
        "google_play_id": "com.greensm.ev",        # 실제 ID로 교체 필요
        "app_store_id": None,
        "app_store_country": "la",
    },
    {
        "app_name": "V-Green",
        "country": "VNM",
        "google_play_id": "com.gsm.customer",      # WBS 데이터수집 시트 참고
        "app_store_id": None,
        "app_store_country": "vn",
    },
]

## 2. Google Play 리뷰 수집

In [ ]:
def scrape_google_play(app_id: str, app_name: str, country: str, count: int = 500) -> pd.DataFrame:
    """
    Google Play 리뷰 수집
    
    Args:
        app_id: Google Play 앱 패키지명
        app_name: 앱 이름 (DB 저장용)
        country: 국가 코드 (ISO 3166-1 alpha-3)
        count: 수집할 최대 리뷰 수
    """
    print(f"\n[Google Play] {app_name} ({app_id}) 수집 시작...")
    
    try:
        result, _ = reviews(
            app_id,
            lang='auto',       # 자동 언어 감지
            country='us',      # 글로벌 리뷰 수집
            sort=Sort.NEWEST,  # 최신순
            count=count,
        )
    except Exception as e:
        print(f"  ❌ 수집 실패: {e}")
        return pd.DataFrame()
    
    if not result:
        print(f"  ⚠️ 리뷰 없음 (앱이 존재하지 않거나 리뷰가 없을 수 있음)")
        return pd.DataFrame()
    
    df = pd.DataFrame(result)
    
    # DB 스키마에 맞게 컬럼 정제
    df_clean = pd.DataFrame({
        'store':       'google_play',
        'app_name':    app_name,
        'country':     country,
        'rating':      df['score'].astype(int),
        'author':      df['userName'],
        'review_date': pd.to_datetime(df['at']).dt.date,
        'content':     df['content'],
    })
    
    # 빈 content 제거
    df_clean = df_clean[df_clean['content'].str.strip().astype(bool)].reset_index(drop=True)
    
    print(f"  ✅ {len(df_clean)}개 리뷰 수집 완료")
    return df_clean

In [ ]:
# Google Play 전체 수집
all_gp_reviews = []

for app in APPS:
    if app['google_play_id']:
        df = scrape_google_play(
            app_id=app['google_play_id'],
            app_name=app['app_name'],
            country=app['country'],
            count=500
        )
        if not df.empty:
            all_gp_reviews.append(df)

if all_gp_reviews:
    gp_df = pd.concat(all_gp_reviews, ignore_index=True)
    print(f"\n📊 Google Play 총 수집: {len(gp_df)}개")
    print(gp_df.groupby(['app_name', 'country']).size())
else:
    print("수집된 리뷰 없음")
    gp_df = pd.DataFrame()

## 3. App Store 리뷰 수집

In [ ]:
def scrape_app_store(app_name: str, app_id: str, country: str, store_country: str, count: int = 500) -> pd.DataFrame:
    """
    App Store 리뷰 수집
    
    Args:
        app_name: 앱 이름
        app_id: App Store 앱 ID (숫자)
        country: 국가 코드 (ISO alpha-3)
        store_country: App Store 국가 코드 (ISO alpha-2, 예: 'us', 'vn')
        count: 수집할 최대 리뷰 수
    """
    print(f"\n[App Store] {app_name} 수집 시작...")
    
    try:
        app = AppStore(country=store_country, app_name=app_name, app_id=app_id)
        app.review(how_many=count)
        result = app.reviews
    except Exception as e:
        print(f"  ❌ 수집 실패: {e}")
        return pd.DataFrame()
    
    if not result:
        print(f"  ⚠️ 리뷰 없음")
        return pd.DataFrame()
    
    df = pd.DataFrame(result)
    
    df_clean = pd.DataFrame({
        'store':       'app_store',
        'app_name':    app_name,
        'country':     country,
        'rating':      df['rating'].astype(int),
        'author':      df['userName'],
        'review_date': pd.to_datetime(df['date']).dt.date,
        'content':     df['review'],
    })
    
    df_clean = df_clean[df_clean['content'].str.strip().astype(bool)].reset_index(drop=True)
    
    print(f"  ✅ {len(df_clean)}개 리뷰 수집 완료")
    return df_clean

In [ ]:
# App Store 전체 수집
all_as_reviews = []

for app in APPS:
    if app['app_store_id']:  # App Store ID가 있는 앱만
        df = scrape_app_store(
            app_name=app['app_name'],
            app_id=app['app_store_id'],
            country=app['country'],
            store_country=app['app_store_country'],
            count=500
        )
        if not df.empty:
            all_as_reviews.append(df)

if all_as_reviews:
    as_df = pd.concat(all_as_reviews, ignore_index=True)
    print(f"\n📊 App Store 총 수집: {len(as_df)}개")
else:
    print("수집된 App Store 리뷰 없음")
    as_df = pd.DataFrame()

## 4. 통합 & DB 적재

In [ ]:
# 전체 통합
all_reviews = pd.concat([df for df in [gp_df, as_df] if not df.empty], ignore_index=True)

print(f"\n📊 전체 수집 요약")
print(f"총 리뷰: {len(all_reviews)}개")
if not all_reviews.empty:
    print(all_reviews.groupby(['app_name', 'store']).size().reset_index(name='count'))
    print(f"\n별점 분포:")
    print(all_reviews['rating'].value_counts().sort_index())

In [ ]:
# Supabase에 적재
if not all_reviews.empty:
    inserted = insert_df(all_reviews, 'app_reviews')
    print(f"\n✅ DB 적재 완료: {inserted}행")
else:
    print("적재할 데이터 없음")

## 5. 수집 결과 확인 (DB에서 조회)

In [ ]:
from src.db import fetch_df

result = fetch_df("""
    SELECT app_name, store, country, 
           COUNT(*) AS total,
           ROUND(AVG(rating), 2) AS avg_rating
    FROM app_reviews
    GROUP BY app_name, store, country
    ORDER BY app_name, store
""")

print("📊 DB 저장 현황")
print(result)